# 🔭 Notebook: Observability with LangSmith — *Part 1 of 2*

*⚠️ This chapter is **advanced and optional** — it is not required to complete the core course, and isn't covered in `setup.md`. Both notebooks need a free account with an external hosted or self-hosted service, on top of everything else in this course.*

*This is the first of two notebooks in this chapter: 1) **LangSmith** (hosted) → 2) Langfuse (self-hosted, via Docker). They cover the same ideas with two different tools - read this one first.*

## 📚 Sources

- [LangSmith: Observability concepts](https://docs.langchain.com/langsmith/observability-concepts)
- [LangSmith: Tracing quickstart](https://docs.langchain.com/langsmith/observability-quickstart)
- [LangSmith: Trace OpenAI-compatible providers](https://docs.langchain.com/langsmith/trace-with-openai-compatible-providers)
- [LangSmith: Trace an LLM application (full tutorial)](https://docs.langchain.com/langsmith/observability-llm-tutorial)

## Why observability?

Every notebook so far has debugged LLM applications the same way: `print()` the messages, read the trace by eye. That works fine for a single call. It stops working once you have an agent that might call a tool five times, or a RAG pipeline with a retrieve → grade → rewrite → generate loop (`07_3_hybrid_rag.ipynb`) - by the time something goes wrong, you're scrolling through a wall of printed dicts trying to reconstruct what actually happened, in what order, and how long each step took.

**Observability** means your application records that structure *as it runs*, instead of you reconstructing it after the fact from print statements. [LangSmith](https://smith.langchain.com/) is LangChain's hosted observability platform: every LLM call, tool call, and chain step gets logged as a structured, timestamped, nested record you can inspect, filter, and search - both in a web UI and, as we'll do here, programmatically.

This is the first genuinely different kind of setup in this course: LangSmith is a separate hosted service from the university's Ollama server, so it needs its own account and its own API key - **your** `LLM_HOST` still points at the university server for the LLM calls themselves; LangSmith just *observes* them from the outside.

## Core concepts

LangSmith organizes data in a small hierarchy:

| Concept | What it is |
|---|---|
| **Project** | A container for all the traces from one application (e.g. "ai-engineering-course") |
| **Trace** | One complete record of a single request end to end - if a user question triggers a tool call and two LLM calls, all of that is one trace |
| **Run** | A single step within a trace (one LLM call, one tool call, one function) - if you know [OpenTelemetry](https://opentelemetry.io/), a run is a *span* |
| **Thread** | Multiple traces linked together as one multi-turn conversation |

A trace is a tree of runs. That tree structure - which step called which, nested how deep, how long each took - is exactly the thing that's tedious to reconstruct from printed output, and exactly what LangSmith captures automatically.

## Setup

Unlike `LLM_HOST`, this needs an account you create yourself:

1. Sign up for free at [smith.langchain.com](https://smith.langchain.com/) (no credit card required for the free tier).
2. Create an API key: **Settings → API Keys → Create API Key**.
3. Add it to your `.env` file, alongside `LLM_HOST`:

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
```

`LANGSMITH_TRACING=true` is the only thing that has to be set for LangChain/LangGraph code to start tracing automatically - no code changes, as you'll see below. `langsmith`'s `Client` and the `@traceable` decorator also read these same two environment variables directly, so `load_dotenv()` is all we need.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads LLM_HOST, LANGSMITH_TRACING, and LANGSMITH_API_KEY from .env

LLM_HOST = os.environ["LLM_HOST"]
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"

# All traces in this notebook go to one named project instead of LangSmith's "default" project
os.environ["LANGSMITH_PROJECT"] = "ai-engineering-course"

In [5]:
from langsmith import Client

ls_client = Client()
print("Connected to LangSmith:", bool(ls_client.info))

Connected to LangSmith: True


## Manual instrumentation

Two building blocks trace *any* Python code, regardless of framework:

- **`wrap_openai`**: wraps an OpenAI-compatible client so every call it makes is automatically logged as a run. Since our Ollama server exposes an OpenAI-compatible API (the same `base_url=".../v1"` pattern from `04_2_intro_structured_outputs.ipynb` and `08_intro_gradio.ipynb`), this works directly against our own setup - no OpenAI account needed.
- **`@traceable`**: wraps any function so its inputs, outputs, and any nested traced calls inside it become one run (or, if it contains other traced calls, a small tree of runs).

Let's build a tiny "library help desk" assistant: a `get_context` step (traced as a `"tool"` run) looks up a fixed FAQ, and an `assistant` step (traced as the parent run) uses that context to answer.

In [6]:
import openai
from langsmith.wrappers import wrap_openai
from langsmith import traceable

# Same OpenAI-compatible client pattern as 04_2 / 08_intro_gradio, wrapped for tracing
client = wrap_openai(openai.OpenAI(base_url=f"{LLM_URL}/v1", api_key="ollama"))

library_faqs = [
    "Books can be borrowed for 21 days and renewed once online.",
    "Overdue books incur a fine of €0.20 per day.",
    "The library is open Monday-Saturday, 8:00-22:00.",
]


@traceable(run_type="tool")
def get_context(question: str) -> str:
    """In a real app this would query a knowledge base or vector store (see 07_1_two_step_rag.ipynb)."""
    return "\n".join(library_faqs)


@traceable
def assistant(question: str) -> str:
    context = get_context(question)
    response = client.chat.completions.create(
        model=LLM_REASONING,
        messages=[
            {"role": "system", "content": f"Answer using only the context below.\n\nContext:\n{context}"},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


answer = assistant("How long can I borrow a book for, and what happens if I'm late?")
print(answer)

Books can be borrowed for 21 days, and overdue books incur a fine of €0.20 per day.


## Verifying the trace

In the LangSmith web UI, you'd open the **Tracing** tab, pick the `ai-engineering-course` project, and click the `assistant` run to see the tree: `assistant` → `get_context` (tool) → `ChatOpenAI` (the actual model call), each with its own inputs, outputs, and latency. We can't screenshot that here - but everything the UI shows is also just data, reachable through the same `Client` we already connected. Let's pull the last few runs back out and reconstruct that same tree from code:

In [7]:
import time

time.sleep(2)  # give LangSmith a moment to finish ingesting the trace we just sent

recent_runs = list(ls_client.list_runs(project_name="ai-engineering-course", limit=5))
for run in recent_runs:
    parent = f" (parent: {run.parent_run_id})" if run.parent_run_id else " (root)"
    print(f"{run.name:15s} [{run.run_type:6s}] status={run.status}{parent}")

ChatOpenAI      [llm   ] status=success (parent: 019f9924-15b5-7600-bf77-ba6c65ec3a00)
get_context     [tool  ] status=success (parent: 019f9924-15b5-7600-bf77-ba6c65ec3a00)
assistant       [chain ] status=success (root)
ChatOpenAI      [llm   ] status=error (parent: 019f9923-79ff-75c3-8e39-8292d85a32fc)
get_context     [tool  ] status=success (parent: 019f9923-79ff-75c3-8e39-8292d85a32fc)


Three runs, one tree: `assistant` (the root - no parent) called `get_context` (a `tool` run, parent = `assistant`), which in turn contains no further calls, and separately `assistant` also triggered a `ChatOpenAI` `llm` run for the actual model call. This *is* what the UI's trace view shows you graphically - we're just reading the same structured data directly.

## Automatic tracing for LangChain and LangGraph

Manual `@traceable` is for arbitrary Python. For anything already built with `create_agent` or `StateGraph` (every agent since `06_2_agents.ipynb`), you don't need to add anything at all - `LANGSMITH_TRACING=true` alone makes LangChain trace every node, tool call, and model call automatically.

In [8]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0)


@tool
def get_temperature(city: str) -> str:
    """Get the current temperature for a city."""
    temperatures = {"New York": "22°C", "London": "15°C", "Tokyo": "18°C"}
    return temperatures.get(city, "Unknown")


# No tracing-related code here at all - this is a plain create_agent, identical to 06_2_agents.ipynb
weather_agent = create_agent(model=llm, tools=[get_temperature])
result = weather_agent.invoke({"messages": [{"role": "user", "content": "What is the temperature in Tokyo?"}]})
print(result["messages"][-1].content)

time.sleep(2)
recent_runs = list(ls_client.list_runs(project_name="ai-engineering-course", limit=6))
print("\nNewest runs (agent's internal graph, traced automatically):")
for run in recent_runs[:4]:
    print(f"  {run.name:15s} [{run.run_type}]")

The current temperature in Tokyo is 18°C.

Newest runs (agent's internal graph, traced automatically):
  ChatOllama      [llm]
  model           [chain]
  LangGraph       [chain]
  ChatOpenAI      [llm]


The agent's entire internal graph - the model node, the tool node, `get_temperature` itself, the surrounding `ChatOllama` call - shows up automatically, with zero tracing-specific code in the agent definition. This is what makes LangSmith practical for the agents and RAG graphs from earlier chapters: you don't retrofit tracing into `06_4_subagents.ipynb` or `07_3_hybrid_rag.ipynb`, it's already there the moment `LANGSMITH_TRACING=true` is set.

## Metadata: tagging runs for filtering

A project quickly accumulates traces from many different calls. **Metadata** - arbitrary key-value pairs attached to a run - is how you later filter or group them (by model, by user, by course chapter, by anything you find useful). Pass it via `@traceable(metadata={...})`, or per-call via `langsmith_extra`.

In [9]:
@traceable(metadata={"model": LLM_REASONING, "course_chapter": "10"})
def tagged_assistant(question: str) -> str:
    response = client.chat.completions.create(
        model=LLM_REASONING,
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content


tagged_assistant("Say hello in one short sentence.")
time.sleep(2)

# Pull the run back and inspect its metadata - the same fields you'd filter by in the UI's Runs table
latest = list(ls_client.list_runs(project_name="ai-engineering-course", limit=1))[0]
custom_metadata = {k: v for k, v in latest.extra.get("metadata", {}).items() if k in ("model", "course_chapter")}
print("Custom metadata on the run:", custom_metadata)

Custom metadata on the run: {'course_chapter': '10', 'model': 'gemma4:26b'}


## Feedback: attaching a score to a run

**Feedback** links a score (and optionally a comment) to a specific run - typically how you record whether a user found a given response helpful. Attaching it requires knowing the run's ID ahead of time, so we generate one ourselves and pass it in via `langsmith_extra`.

In [10]:
import uuid

run_id = str(uuid.uuid4())
answer = assistant(
    "What are the library's opening hours?",
    langsmith_extra={"run_id": run_id},
)
print(answer)

ls_client.create_feedback(run_id, key="user-score", score=1.0, comment="Correct and concise.")

time.sleep(2)
feedback = list(ls_client.list_feedback(run_ids=[run_id]))
print("Feedback on this run:", [(f.key, f.score, f.comment) for f in feedback])

The library is open Monday-Saturday, 8:00-22:00.
Feedback on this run: [('user-score', 1.0, 'Correct and concise.')]


In a real app, `assistant(...)` and `create_feedback(...)` would live in two different places entirely: the assistant call happens when the user asks their question; the feedback call happens later, from wherever your app captures a 👍/👎 click (e.g. a Gradio button, as in `08_intro_gradio.ipynb`) - `run_id` is what ties the two together across that gap.

## Production: monitoring (conceptual)

Everything above works at the scale of "one trace, inspect it by hand." In production, with real traffic, you stop reading individual traces and start watching aggregates instead: the **Monitoring** tab in the LangSmith UI charts trace volume, latency, error rate, and feedback scores over time for a whole project, and lets you **group by** a metadata key (e.g. compare two model versions tagged via `metadata={"model": ...}`, exactly like we tagged `tagged_assistant` above) to spot regressions before they pile up as individual complaints.

We won't set this up here - it needs real, sustained traffic to produce a chart worth looking at, not a handful of notebook calls - but it's the natural next step once metadata tagging (which we just did) is already in place.

## Exercise: Trace and tag a RAG-style call

Write a `@traceable` function `search_and_answer(question)` that:
1. Calls a `@traceable(run_type="retriever")` helper function that returns 2-3 fixed strings of your choosing (any topic).
2. Passes them as context into an LLM call, same pattern as `assistant` above.
3. Tags the outer function with `metadata={"exercise": "10_1"}`.

Then use `ls_client.list_runs(...)` to confirm: the retriever run exists, is nested under your outer function, and the metadata is attached correctly.

In [11]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

```python
@traceable(run_type="retriever")
def retrieve(query: str) -> list[str]:
    return [
        "The Eiffel Tower was completed in 1889.",
        "It was originally intended as a temporary structure.",
    ]


@traceable(metadata={"exercise": "10_1"})
def search_and_answer(question: str) -> str:
    context = retrieve(question)
    response = client.chat.completions.create(
        model=LLM_REASONING,
        messages=[
            {"role": "system", "content": f"Answer using only this context:\n{chr(10).join(context)}"},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


search_and_answer("When was the Eiffel Tower completed?")
time.sleep(2)

recent_runs = list(ls_client.list_runs(project_name="ai-engineering-course", limit=3))
for run in recent_runs:
    print(run.name, run.run_type, run.extra.get("metadata", {}).get("exercise"))
```

</details>

---

**Next up:** `10_2_langfuse.ipynb` - the same ideas (traces, runs, metadata, feedback), on a self-hosted, open-source alternative you run yourself via Docker.